In [1]:
import os, glob, copy, zipfile, random
from PIL import Image
from tqdm import tqdm
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
from torchvision import models, transforms
from sklearn.model_selection import train_test_split

# deterministic behavior for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False



In [2]:
mean = (0.485, 0.456, 0.406)  # ImageNet mean
std = (0.229, 0.224, 0.225)  # ImageNet std
batch_size = 32
lr = 0.001
epochs = 2  # increased from 1 to 2 for modest improvement



In [3]:
base_path = "/kaggle/input/dogs-vs-cats-redux-kernels-edition"

train_dir = os.path.join(base_path, "train")
test_dir = os.path.join(base_path, "test", "unknown")  # test images are here

train_list = glob.glob(os.path.join(train_dir, "*", "*.jpg"))
test_list = glob.glob(os.path.join(test_dir, "*.jpg"))

train_list, val_list = train_test_split(train_list, test_size=0.2, random_state=42)

subset_frac = 0.12  # slightly larger subset (12% instead of 10%)
subset_size = int(len(train_list) * subset_frac)
if subset_size < len(train_list):
    np.random.seed(42)
    np.random.shuffle(train_list)
    train_list = train_list[:subset_size]

print(f"Train images  : {len(train_list)}")
print(f"Validation images : {len(val_list)}")
print(f"Test images   : {len(test_list)}")



Train images  : 2160
Validation images : 4500
Test images   : 2500


In [4]:
train_transforms = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ]
)

val_transforms = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ]
)




In [5]:
class CatDogDataset(data.Dataset):
    def __init__(self, file_list, transform=None):
        self.file_list = file_list
        self.transform = transform

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_path = self.file_list[idx]
        img = Image.open(img_path).convert("RGB")
        img = self.transform(img) if self.transform else img

        label_str = os.path.basename(img_path).split(".")[0]  # 'dog' or 'cat'
        label = 1 if label_str == "dog" else 0
        return img, label




In [6]:
train_dataset = CatDogDataset(train_list, transform=train_transforms)
val_dataset = CatDogDataset(val_list, transform=val_transforms)

train_loader = data.DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, num_workers=2
)
val_loader = data.DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False, num_workers=2
)



In [7]:
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

num_classes = 2
model.fc = nn.Linear(model.fc.in_features, num_classes)

for name, param in model.named_parameters():
    param.requires_grad = name.startswith("layer4")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(trainable_params, lr=lr)



Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 104MB/s]


In [8]:
best_accuracy = 0.0
best_state = copy.deepcopy(model.state_dict())

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    epoch_acc = 0.0

    for imgs, lbls in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
        imgs = imgs.to(device)
        lbls = lbls.to(device)

        outputs = model(imgs)
        loss = criterion(outputs, lbls)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        acc = (outputs.argmax(dim=1) == lbls).float().mean()
        epoch_acc += acc.item()
        epoch_loss += loss.item()

    epoch_acc /= len(train_loader)
    epoch_loss /= len(train_loader)

    model.eval()
    val_acc = 0.0
    val_loss = 0.0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs = imgs.to(device)
            lbls = lbls.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, lbls)
            acc = (outputs.argmax(dim=1) == lbls).float().mean()
            val_acc += acc.item()
            val_loss += loss.item()
    val_acc /= len(val_loader)
    val_loss /= len(val_loader)

    print(
        f"Epoch {epoch+1}: train loss {epoch_loss:.4f}, acc {epoch_acc:.4f} | val loss {val_loss:.4f}, acc {val_acc:.4f}"
    )

    if val_acc > best_accuracy:
        best_accuracy = val_acc
        best_state = copy.deepcopy(model.state_dict())

os.makedirs("../working", exist_ok=True)
torch.save(best_state, "../working/model.pth")



Epoch 1: train loss 0.2236, acc 0.9058 | val loss 0.0929, acc 0.9634


Epoch 2: train loss 0.1013, acc 0.9600 | val loss 0.0995, acc 0.9632


In [9]:
try:
    checkpoint = torch.load("../working/model.pth", map_location=device)
    model.load_state_dict(checkpoint)
    print("Loaded best model checkpoint.")
except FileNotFoundError:
    print("Checkpoint not found – using current model weights.")

model.eval()
id_list = []
pred_list = []

with torch.no_grad():
    for test_path in tqdm(test_list, desc="Predicting"):
        img = Image.open(test_path).convert("RGB")
        img_tensor = val_transforms(img).unsqueeze(0).to(device)
        output = model(img_tensor)
        prob_dog = F.softmax(output, dim=1)[0, 1].item()
        img_id = int(os.path.splitext(os.path.basename(test_path))[0])
        id_list.append(img_id)
        pred_list.append(prob_dog)



Loaded best model checkpoint.


Predicting: 100%|██████████| 2500/2500 [00:10<00:00, 245.22it/s]


In [10]:
sample_sub_path = os.path.join(base_path, "sample_submission.csv")
submit = pd.read_csv(sample_sub_path)

submit.set_index("id", inplace=True)
submit.loc[id_list, "label"] = pred_list
submit.reset_index(inplace=True)

out_path = "/kaggle/working/submission.csv"
submit.to_csv(out_path, index=False)
print(f"Submission written to {out_path}")

Submission written to /kaggle/working/submission.csv
